# Rodrigo Baruch Rivera Rico
# Curso de Optimización 
# Tarea 3

| Descripción:                         | Fechas                   |
|--------------------------------------|--------------------------|
| Fecha de publicación del documento:  | **Septiembre  4, 2026**  |
| Fecha límite de entrega de la tarea: | **Septiembre 13, 2026**  |

## Indicaciones


Este notebook debe ser un reporte de la tarea, por lo cual:

- Para los ejercicios teóricos puede escribir la respuesta en una celda Markdown o
  insertar una imagen con la respuesta si se ve claran las palabras.
- Para los ejercicios prácticos hay que escribir el código de las funciones que implementan
  los algoritmos y los casos de prueba en C o C++. Ejecute las celdas para que queden registrados
  los resultados.
- Cuando se dice que una función de C tiene que devolver una serie de valores,
  puede definir algunos de los argumentos de la función como apuntadores o indicar que
  se pasan los valores de una variable por referencia, para que puedan ser modificados
  dentro de la función. También podría definir una estructura de datos, pero tome en cuenta
  que para cada algoritmo tendría que definir una estructura de datos diferente. La elección
  de como define las entradas y salidas de una función es libre.
- En caso que se pida generar una gráfica, puede exportar los datos que se necesitan a
  un archivo CSV, Excel o Json, y en usar otro Notebook para leer los datos y generar la gráfica
  usando Julia, Python o R.
- Hay que subir al Classroom el notebook y los archivos necesarios para ejecutar las funciones.
- También hay que exportar el notebook a PDF y agregarlo al Classroom para poner anotaciones
  y dar la retroalimentación correspondiente de la tarea.

----

### Información adicional para esta tarea

Revise el PDF `archivos_binarios.pdf` 
que tiene la explicación del formato de los archivos binarios que se van
a utilizar para cargar la información de las matrices y vectores de los
sistemas de ecuaciones que tienen que resolver.

El notebook `archivos_binarios.ipynb` y el código que pueden usar 
para el manejo de los archivo binarios se encuentra el archivo Zip
`ayudantia01.zip`

----

In [45]:
from itertools import chain
from pathlib import Path

carpetas_ignoradas = {"ayudantia01"}
archivos_ignorados = {"prueba.c"}

for archivo in chain(Path(".").rglob("*.[ch]"), Path(".").rglob("*.bin")):
    if not carpetas_ignoradas.intersection(archivo.parts) and archivo.name not in archivos_ignorados:
        print(archivo)

arrays\array1D.c
arrays\array1D.h
arrays\array2D.c
arrays\array2D.h
ejercicio1\Ejercicio1.c
ejercicio2\Ejercicio2.c
ejercicio3\Ejercicio3.c
sol_ecu_lin\sol_ecu_lin.c
sol_ecu_lin\sol_ecu_lin.h
datosTarea03\matA005.bin
datosTarea03\matA010.bin
datosTarea03\matA025.bin
datosTarea03\matA050.bin
datosTarea03\matA500.bin
datosTarea03\matL005.bin
datosTarea03\matL010.bin
datosTarea03\matL025.bin
datosTarea03\matL050.bin
datosTarea03\matL500.bin
datosTarea03\matU005.bin
datosTarea03\matU010.bin
datosTarea03\matU025.bin
datosTarea03\matU050.bin
datosTarea03\matU500.bin
datosTarea03\vecb005.bin
datosTarea03\vecb010.bin
datosTarea03\vecb025.bin
datosTarea03\vecb050.bin
datosTarea03\vecb500.bin


In [ ]:
# %load arrays/array1D.c
#include "array1D.h"

// Reserva memoria para un arreglo de tamaño n
Array1d *array1d_alloc(size_t n) {
    // Reservamos memoria
    Array1d *array = (Array1d *) malloc(sizeof(Array1d));
    if (!array) { 
        return NULL; 
    }
    
    array->data = (double *) calloc(n, sizeof(double));
    if(array->data==NULL) {
        free(array);
        return NULL;
    }
    array->ndim = 1;
    array->n    = n;

    return(array);
}

// Crea un arreglo 1D a partir de la información en un archivo binario
Array1d *readArray1d(const char *cfullname) {
    uint64_t n;
    FILE    *f1 = fopen(cfullname, "rb");
    
    if(!f1) return(NULL);
    fread(&n, sizeof(uint64_t), 1, f1);

    Array1d *array = array1d_alloc(n);
    if(!array) {
        return NULL;
    }

    fread(array->data, sizeof(double), n, f1);
    fclose(f1);

    return(array);
}

// Escribe las entradas de un arreglo en un archivo binario
int writeArray1d(Array1d *array, const char *cfullname) {
    int     n;
    FILE   *f1 = fopen(cfullname, "wb");
    
    if(!f1) return(1);
    fwrite(&(array->n), sizeof(size_t), 1, f1);
    fwrite(array->data, sizeof(double), array->n, f1);
    fclose(f1);
    return(0);
}


// Libera la memoria del arreglo unidimensional
void freeArray1d(Array1d *array) {
    free(array->data);
    free(array);
}

// Imprime los elementos de un vector usando el formato especificado
// en la cadena format.
// nshow indica la cantidad de valores que se imprimen del inicio y del final
// del arreglo. Esto para evitar imprimir todo el arreglo cuando es muy grande.
void printArray1d(Array1d *array, const char *format, int nshow)  {
    size_t   i;
    printf("[");
    if(nshow==0 || (2*nshow)>=array->n) {
        for(i=0; i<array->n; i++) {
            printf(format, array->data[i]);
        }
    }
    else {
        for(i=0; i<nshow; i++) {
            printf(format, array->data[i]);
        }
        printf(" ... ");
        for(i=array->n-nshow; i<array->n; i++) {
            printf(format, array->data[i]);
        }
    }
    printf("]\n");
}




In [ ]:
# %load arrays/array2D.c
#include "array2D.h"


/*
* Reserva memoria para un arreglo 2D de tamaño nr x nc
* calloc() asegura que cada byte del bloque asignado contenga el valor 0
*/
Array2d *array2d_alloc(int nr, int nc) {
    // Reservamos memoria
    Array2d* array = (Array2d *) malloc(sizeof(Array2d));
    if(!array) {
        return NULL;
    }
    array->data = (double **) malloc( (nr)*sizeof(double *));
    if(array->data==NULL) {
        free(array);
        return(NULL);
    }
    array->data[0] = (double *) calloc(nr*nc, sizeof(double));
    if(array->data[0]==NULL) {
        free(array->data);
        free(array);
        return(NULL);
    }
    array->rows = nr;
    array->cols = nc;
    array->ndim = 2;
    for(int i=1; i<nr; ++i) 
        array->data[i] = array->data[i-1] + nc;
    return(array);
}

// Lectura de las entradas de un arreglos 2D almacenadas en un archivo binario.
// Devuelve NULL si no se pudo abrir el archivo.
Array2d *readArray2d(const char *cfile) {
    Array2d *array;
    uint64_t nr, nc;
    FILE    *f1 = fopen(cfile, "rb");
    
    if(!f1)  return NULL;
    fread(&nr, sizeof(uint64_t), 1, f1);
    fread(&nc, sizeof(uint64_t), 1, f1);
    array = array2d_alloc(nr, nc);
    if(!array) return NULL;

    fread(array->data[0], sizeof(double), nr*nc, f1);
    fclose(f1);
    return(array);
}

// Almacena las entradas del arreglo 2D en un archivo binario.
// Devuelve 0 en caso de exito y 1 si no.
int writeArray2d(Array2d *array, const char *cfullname) {
    FILE       *f1 = fopen(cfullname, "wb");
    
    if(!f1) return(1);
    int  nr = array->rows,  nc=array->cols;
    fwrite(&nr, sizeof(int), 1, f1);
    fwrite(&nc, sizeof(int), 1, f1);
    fwrite(array->data[0], sizeof(double), nr*nc, f1);
    fclose(f1);
    return(0);
}


// Libera la memoria del arreglo bidimensional
void freeArray2d(Array2d *array) {
    free(array->data[0]);
    free(array->data);
    free(array);
}

// Imprime la fila i-esima del arreglo 
void printRow(double **data, int i, int nc, const char *format, int nview) {
    printf("[");
    if((2*nview+1)<nc) {
        for(int j=0; j<nview; ++j) 
            printf(format, data[i][j]);
        printf(" ... ");
        for(int j=nc-nview; j<nc; ++j) 
            printf(format, data[i][j]);
    }
    else {
        for(int j=0; j<nc; ++j) 
            printf(format, data[i][j]);
    }
    printf("]\n");
}

// Imprime en la consola las entradas del arraglo 2D usando el formato indicado 
// en la cadena format.
// nshow indica la cantidad de filas que se imprimen del inicio y del final
// del arreglo, así como la cantidad de valores de valores que se imprimen
// al inicio y final de cada fila.
// Esto para evitar imprimir todo el arreglo cuando es muy grande.
void printArray2d(Array2d *array, const char *format, int nview) {
    int i, j, nr=array->rows, nc=array->cols; 

    if((2*nview+1)<nr ) {
        for(i=0; i<nview; ++i) 
            printRow(array->data, i, nc, format, nview);
        printf("... \n");
        for(i=nr-nview; i<nr; ++i) 
            printRow(array->data, i, nc, format, nview);
    }
    else {
        for(i=0; i<nr; ++i) 
            printRow(array->data, i, nc, format, nview);
    }

}





In [ ]:
# %load sol_ecu_lin/sol_ecu_lin.c
#include "../arrays/array1D.h"
#include "../arrays/array2D.h"
#include "sol_ecu_lin.h"
#include <math.h>
#include <stdlib.h>

Array1d *forwardSubstitution(Array2d *L, Array1d *b, double tol){

    Array1d *x = NULL;

    size_t n = b->n;

    x = array1d_alloc(n);
    if(!x){
        printf("\n%s","No hay memoria suficiente para guardar la solucio'n del sistema.");
        return NULL;}

    // Solución al sistema Lx = b
    double suma;
    for(size_t i=0;i<n;i++){

        if(fabs(L->data[i][i]) < tol){
            printf("\n%s","Divisio'n entre cero.");
            freeArray1d(x);
            return NULL;
        }

        suma = 0;
        for(size_t j=0;j<i;j++){
            suma += (L->data[i][j]) * (x->data[j]);
        }
        
        x->data[i] = ((b->data[i]) - suma)/(L->data[i][i]);
    }

    return x;
}

Array1d *backwardSubstitution(Array2d *U, Array1d *b, double tol){

    Array1d *x = NULL;

    size_t n = b->n;

    x = array1d_alloc(n);
    if(!x){
        printf("\n%s","No hay memoria suficiente para guardar la solucio'n del sistema.");
        return NULL;}

    // Solución al sistema Ux = b
    double suma;
    for(size_t i=n;i-- > 0; ){

        if(fabs(U->data[i][i]) < tol){
            printf("\n%s","Divisio'n entre cero.");
            freeArray1d(x);
            return NULL;
        }

        suma = 0;
        for(size_t j=i+1;j<n;j++){
            suma += (U->data[i][j]) * (x->data[j]);
        }
        
        x->data[i] = ((b->data[i]) - suma)/(U->data[i][i]);
    }

    return x;
}

outLU *LU(Array2d *A, double tol){

    Array2d *L=NULL,*U=NULL;
    Array1d *p=NULL;

    /* Validar A */
    if(!A){ 
        printf("\n%s","Matriz no válida.");
        return NULL;
    }
    if (A->rows != A->cols) {
        printf("\n%s","No es matriz cuadrada.");
        return NULL;
    }
    size_t n = A->rows; //Matriz cuadrada de tamaño nxn


    /* Memoria para salida de la función */
    outLU *out = malloc(sizeof *out);
    if(!out){
        printf("\n%s","No hay memoria suficiente para guardar la salida de la funcio'n");
        return NULL;
    }
    out->L=NULL;
    out->U=NULL;
    out->p=NULL;

    /* Inicializamos L
     * Matriz identidad de tamaño nxn 
     */
    L = array2d_alloc(n,n);
    if(!L){
        printf("\n%s","No hay memoria suficiente para crear la matriz L.");
        free(out);
        return NULL;
    }
    for(size_t i=0;i<n;i++){
        L->data[i][i] = 1.0;
    }

    /* Inicializamos U
     * Copia de la matriz A
     */
    U = array2d_alloc(n,n);
    if(!U){
        printf("\n%s", "No hay memoria suficiente para crear la matriz U");
        freeArray2d(L); free(out);
        return NULL;
    }
    for(size_t i=0;i<n;i++){
        for(size_t j=0;j<n;j++){
            U->data[i][j] = A->data[i][j];
        }
    }

    /* Inicializamos p
     * Arreglo 1D donde registramos los intercambios de filas
     */
    p = array1d_alloc(n);
    if(!p){
        printf("\n%s","No hay memoria suficiente para crear el arreglo de permutacio'n p.");
        freeArray2d(L); freeArray2d(U); free(out);
        return NULL;
    }
    for(size_t i=0;i<n;i++){
        p->data[i]=i;
    }

    /* Bucle principal */
    for(size_t k=0;k<n-1;k++){
        /* Buscar pivote */
        int r;
        r = k;
        for (size_t i=k+1;i<n;i++) {
            if (fabs(U->data[i][k]) > fabs(U->data[r][k])) {
                r = i;
            }
        }

        /* Comprobar pivote */
        if(fabs(U->data[r][k]) < tol){
            out->L = L;
            out->U = U;
            out->p = p;
            out->res = SINGULAR;
            return out;
        }

        /* Intercambiar filas de U 
         * Registrar intercambio en p
         */
        if(r != k){
            for(size_t j=0;j<n;j++){
                double temp = U->data[r][j];
                U->data[r][j] = U->data[k][j];
                U->data[k][j] = temp;
            }
            int temp = p->data[r];
            p->data[r] = p->data[k];
            p->data[k] = temp;

            /* Intercambiar multiplicadores ya calculados */
            for(size_t j=0;j<k;j++){
                double temp = L->data[r][j];
                L->data[r][j] = L->data[k][j];
                L->data[k][j] = temp;
            }
        }

        /* Eliminación */
        for(size_t i=k+1;i<n;i++){
            L->data[i][k] = U->data[i][k] / U->data[k][k];
        }

        for(size_t i=k+1;i<n;i++){
            for(size_t j=k+1;j<n;j++){
                U->data[i][j] -= L->data[i][k] * U->data[k][j];
            }
        }
        
        for(size_t i=k+1;i<n;i++){
            U->data[i][k]=0;
        }
    }

    /* Comprobar último pivote */
    if (fabs(U->data[n-1][n-1]) < tol) {
        out->L = L;
        out->U = U;
        out->p = p;
        out->res = SINGULAR;
        return out;
    }

    /* Éxito */
    out->L = L;
    out->U = U;
    out->p = p;
    out->res = EXITO;
    return out;
}

Array1d *solveLU(Array2d *L, Array2d *U, Array1d *p, Array1d *b, double tol){

    Array1d *b_hat=NULL,*y=NULL,*x=NULL;
    size_t n = b->n;

    /* Vector de valores independientes reordenado 
     * b_hat es la representación de Pb
     */
    b_hat = array1d_alloc(n);
    if(!b_hat){
        printf("\n%s","Memoria insuficiente.");
        return NULL;
    }
    for(size_t i=0;i<n;i++){
        size_t idx = p->data[i];
        b_hat->data[i] = b->data[idx];
    }

    /* Sustitución hacia adelante para Ly = b_hat 
     * con Ax = y
     */
    y = forwardSubstitution(L,b_hat,tol);
    if(!y){
        printf("\n%s","El sistema no tiene solucio'n u'nica.");
        freeArray1d(b_hat);
        return NULL;
    }

    /* Sustitucion hacia atrás para Ux = y */
    x = backwardSubstitution(U,y,tol);
    if(!x){
        printf("\n%s","El sistema no tiene solucio'n u'nica.");
        freeArray1d(b_hat); freeArray1d(y);
        return NULL;
    }

    freeArray1d(b_hat); freeArray1d(y);
    return x;
}

## Ejercicio 1 (2.5 puntos)

Calcular la solución de sistemas con matrices triangulares inferiores.

1. Programar la función  ``forwardSubstitution`` que aplica el método de 
   sustitución hacia adelante (descrito en las diapositivas de la clase 6)
   para resolver el sistema $\mathbf{L}\mathbf{x} = \mathbf{b}$, 
   donde $\mathbf{L}$ es una matriz triangular inferior.

**Entradas de la función:**
    
- La matriz $\mathbf{L}$,
- El arreglo $\mathbf{b}$ y 
- una tolerancia $\tau$ para prevenir la división entre un número muy pequeño. 

**Salida de la función:**
    
- El arreglo $\mathbf{x}$ que es solución del sistema si el algoritmo termina 
  exitósamente o **NULL** si no se pudo calcular la solución.
    
2. Escriba una función que reciba como parámetros los nombres de
   dos archivos `.bin`,  uno que tiene la información de 
   una matriz triangular inferior $\mathbf{L}$
   y el otro con la información del vector de términos independientes $\mathbf{b}$. 
   
   Lea los archivos para crear la matriz $\mathbf{L}$ y el vector $\mathbf{b}$.
   Imprima  el tamaño de la matriz y el tamaño del vector $\mathbf{b}$.

   Llame a la función ``forwardSubstitution`` para resolver el sistema 
   $\mathbf{L}\mathbf{x} = \mathbf{b}$, 
   usando como tolerancia $\tau = \epsilon_m^{2/3}$, donde $\epsilon_m$ es el épsilon máquina. 
   Si el sistema tuvo solución, imprima los primeros y últimos **3 elementos**
   del vector solución. En caso contrario, imprima el mensaje de
   que el sistema no tiene solución única. 
   
   Imprima el valor del error $\|\mathbf{L}\mathbf{x} - \mathbf{b}\|$ si existe la solución. 
   En caso contrario, haga que la función imprima un mensaje que 
   indique que la matriz es singular. 
    
4. Pruebe la función del punto anterior usando los datos del archivo ``datosTarea03.zip``,
   usando las parejas de archivos:
   
| Matriz      | Vector          |
|-------------|-----------------|
| matL005     |  vecb005        |
| matL010     |  vecb010        |
| matL025     |  vecb025        |
| matL050     |  vecb050        |
| matL500     |  vecb500        |

### Solución:

In [ ]:
# %load ejercicio1/Ejercicio1.c
#include "../arrays/array1D.h"
#include "../arrays/array2D.h"
#include "../sol_ecu_lin/sol_ecu_lin.h"
#include <math.h>

#define OK 1
#define ERROR_METHOD 2
#define ERROR_READ_BIN 4
#define ERROR_INPUT 8

// Devuelve el épsilon de la máquina
double epsilon(void);

int main(int argc, char **argv){

    Array1d *b=NULL, *x=NULL;
    Array2d *L=NULL;

    double tol = pow(epsilon(),2.0/3.0);

    if(argc<3) {
        printf("Hay que proporcionar dos para'metros:");
        printf("1. El nombre del archivo binario de un arreglo 1D");
        printf("2. El nombre del archivo binario de un arreglo 2D");
        return ERROR_INPUT;
    } 

    b = readArray1d(argv[1]);
    if(!b){return ERROR_READ_BIN;}
    
    L = readArray2d(argv[2]);
    if(!L){
        freeArray1d(b);
        return ERROR_READ_BIN;
    }
   
    /*Tamaño de b*/
    printf("\nEl tamano del vector b es <%zu>",b->n);
    /*Elementos de b*/
    // printf("\n");
    // printArray1d(b, "% 6.2f  ", 3);

    /*Tamaño de L*/
    printf("\nLa matriz L tiene <%zu> filas y <%zu> columnas", L->rows, L->cols);
    /*Elementos de L*/
    // printf("\n");
    // printArray2d(L, "% 6.2f  ", 3);

    /*Obtenemos la solución*/
    x = forwardSubstitution(L, b, tol);
    if(!x){
        printf("\n%s","El sistema no tiene solucio'n u'nica.");
        freeArray1d(b);
        freeArray2d(L);    
        return ERROR_METHOD;
    }

    /*Elementos de x*/
    printf("\nLa solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:\n");
    printArray1d(x, "% 6.2f  ", 3);

    /*Error residual*/
    // Vector residual: Lx-b
    double r_i; /**< Entrada i-ésima del vector residual */
    double err_residual = 0;

    for(size_t i=0;i<L->rows;i++){
        r_i = 0;
        for(size_t j=0;j<L->cols;j++){
            r_i += L->data[i][j] * x->data[j];
        }
        r_i -= b->data[i];
        err_residual += r_i*r_i;
    }
    err_residual = sqrt(err_residual);

    printf("\nError residual ||Lx-b|| = %e",err_residual);

    freeArray2d(L); freeArray1d(b); freeArray1d(x);

    return OK;
}

double epsilon(void){
    
    double eps = 0.5;
    double unit = 1.0;
    double val = unit + eps;

    while(val > unit){
        eps/=2;
        val = unit + eps;
    }
    eps = 2*eps;

    return eps;
}

In [47]:
!gcc ejercicio1\Ejercicio1.c arrays\array1D.c arrays\array2D.c sol_ecu_lin\sol_ecu_lin.c -lm -o ejercicio1\ex1

In [48]:
!ejercicio1\ex1 datosTarea03\vecb005.bin datosTarea03\matL005.bin


El tamano del vector b es <5>
La matriz L tiene <5> filas y <5> columnas
La solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:
[-37.44   63.50   19.11  -52.35   21.20  ]

Error residual ||Lx-b|| = 1.776357e-15


In [49]:
!ejercicio1\ex1 datosTarea03\vecb010.bin datosTarea03\matL010.bin


El tamano del vector b es <10>
La matriz L tiene <10> filas y <10> columnas
Divisio'n entre cero.
El sistema no tiene solucio'n u'nica.


In [50]:
!ejercicio1\ex1 datosTarea03\vecb025.bin datosTarea03\matL025.bin


El tamano del vector b es <25>
La matriz L tiene <25> filas y <25> columnas
La solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:
[-244.78   1214.34   365.70   ...  906.01  -370.09   297.76  ]

Error residual ||Lx-b|| = 2.120367e-13


In [51]:
!ejercicio1\ex1 datosTarea03\vecb050.bin datosTarea03\matL050.bin


El tamano del vector b es <50>
La matriz L tiene <50> filas y <50> columnas
La solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:
[-35.51    4.58   31.38   ... -96.64  -130.05   68.84  ]

Error residual ||Lx-b|| = 6.109738e-14


In [52]:
!ejercicio1\ex1 datosTarea03\vecb500.bin datosTarea03\matL500.bin


El tamano del vector b es <500>
La matriz L tiene <500> filas y <500> columnas
La solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:
[-30.61  -38.09   61.09   ... -925.84  -328.81  -67.14  ]

Error residual ||Lx-b|| = 4.790422e-13


```







```
---

## Ejercicio 2 (2.5 puntos)

Calcular la solución de sistemas con matrices triangulares superiores.

1. Programar la función  ``backwardSubstitution`` que aplica el método de 
   sustitución hacia atrás (descrito en las diapositivas de la clase 6)
   para resolver el sistema $\mathbf{U}\mathbf{x} = \mathbf{b}$, 
   donde $\mathbf{U}$ es una matriz triangular superior.

**Entradas de la función:**
    
- La matriz $\mathbf{U}$,
- El arreglo $\mathbf{b}$ y 
- una tolerancia $\tau$ para prevenir la división entre un número muy pequeño. 

**Salida de la función:**
    
- El arreglo $\mathbf{x}$ que es solución del sistema si el algoritmo termina 
  exitósamente o None si no se pudo calcular la solución.

2. Análogo al Ejercicio 1, 
   escriba un programa para resolver el sistema  $\mathbf{U}\mathbf{x} = \mathbf{b}$
   siguiendo las mismas indicaciones y pruebe el programa
   usando los datos del archivo ``datosTarea03.zip`` con las
   siguiente parejas de archivos:
   
 
| Matriz      | Vector          |
|-------------|-----------------|
| matU005     |  vecb005        |
| matU010     |  vecb010        |
| matU025     |  vecb025        |
| matU050     |  vecb050        |
| matU500     |  vecb500        |

### Solución:

In [ ]:
# %load ejercicio2/Ejercicio2.c
#include "../arrays/array1D.h"
#include "../arrays/array2D.h"
#include "../sol_ecu_lin/sol_ecu_lin.h"
#include <math.h>

#define OK 1
#define ERROR_METHOD 2
#define ERROR_READ_BIN 4
#define ERROR_INPUT 8

// Devuelve el épsilon de la máquina
double epsilon(void);

int main(int argc, char **argv){

    Array1d *b=NULL, *x=NULL;
    Array2d *U=NULL;

    double tol = pow(epsilon(),2.0/3.0);

    if(argc<3) {
        printf("Hay que proporcionar dos para'metros:");
        printf("1. El nombre del archivo binario de un arreglo 1D");
        printf("2. El nombre del archivo binario de un arreglo 2D");
        return ERROR_INPUT;
    } 

    b = readArray1d(argv[1]);
    if(!b){return ERROR_READ_BIN;}
    
    U = readArray2d(argv[2]);
    if(!U){
        freeArray1d(b);
        return ERROR_READ_BIN;
    }
   
    /*Tamaño de b*/
    printf("\nEl tamano del vector b es <%zu>",b->n);
    /*Elementos de b*/
    // printf("\n");
    // printArray1d(b, "% 6.2f  ", 3);

    /*Tamaño de U*/
    printf("\nLa matriz U tiene <%zu> filas y <%zu> columnas", U->rows, U->cols);
    /*Elementos de U*/
    // printf("\n");
    // printArray2d(U, "% 6.2f  ", 3);

    /*Obtenemos la solución*/
    x = backwardSubstitution(U, b, tol);
    if(!x){
        printf("\n%s","El sistema no tiene solucio'n u'nica.");
        freeArray1d(b);
        freeArray2d(U);    
        return ERROR_METHOD;
    }

    /*Elementos de x*/
    printf("\nLa solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:\n");
    printArray1d(x, "% 6.2f  ", 3);

    /*Error residual*/
    // Vector residual: Ux-b
    double r_i; /**< Entrada i-ésima del vector residual */
    double err_residual = 0;

    for(size_t i=0;i<U->rows;i++){
        r_i = 0;
        for(size_t j=0;j<U->cols;j++){
            r_i += U->data[i][j] * x->data[j];
        }
        r_i -= b->data[i];
        err_residual += r_i*r_i;
    }
    err_residual = sqrt(err_residual);

    printf("\nError residual ||Ux-b|| = %e",err_residual);

    freeArray2d(U); freeArray1d(b); freeArray1d(x);

    return OK;
}

double epsilon(void){
    
    double eps = 0.5;
    double unit = 1.0;
    double val = unit + eps;

    while(val > unit){
        eps/=2;
        val = unit + eps;
    }
    eps = 2*eps;

    return eps;
}

In [57]:
!gcc ejercicio2\Ejercicio2.c arrays\array1D.c arrays\array2D.c sol_ecu_lin\sol_ecu_lin.c -lm -o ejercicio2\ex2

In [58]:
!ejercicio2\ex2 datosTarea03\vecb005.bin datosTarea03\matU005.bin


El tamano del vector b es <5>
La matriz U tiene <5> filas y <5> columnas
La solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:
[ -2.21   14.41  -14.82    0.60   -7.99  ]

Error residual ||Ux-b|| = 8.881784e-15


In [59]:
!ejercicio2\ex2 datosTarea03\vecb010.bin datosTarea03\matU010.bin


El tamano del vector b es <10>
La matriz U tiene <10> filas y <10> columnas
La solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:
[ -1.03    8.53   -4.69   ...   3.28    1.96    4.84  ]

Error residual ||Ux-b|| = 1.464821e-14


In [60]:
!ejercicio2\ex2 datosTarea03\vecb025.bin datosTarea03\matU025.bin


El tamano del vector b es <25>
La matriz U tiene <25> filas y <25> columnas
Divisio'n entre cero.
El sistema no tiene solucio'n u'nica.


In [61]:
!ejercicio2\ex2 datosTarea03\vecb050.bin datosTarea03\matU050.bin


El tamano del vector b es <50>
La matriz U tiene <50> filas y <50> columnas
La solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:
[  4.44    3.04   -8.21   ...  -0.22   -0.90    0.60  ]

Error residual ||Ux-b|| = 3.321906e-13


In [62]:
!ejercicio2\ex2 datosTarea03\vecb500.bin datosTarea03\matU500.bin


El tamano del vector b es <500>
La matriz U tiene <500> filas y <500> columnas
La solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:
[ -2.21    0.06   -4.01   ...   0.68    0.14   -0.10  ]

Error residual ||Ux-b|| = 1.615198e-11


```







```


---


## Ejercicio 3 (5 puntos)

Programar el algoritmo de factorización LU con pivoteo parcial y
probarlo resolviendo un sistema de ecuaciones lineales de acuerdo al 
Algoritmo 1 de la Clase 8.


1. Programe la función que calcula la factorización LU
   de acuerdo al Algoritmo 1:

**Entrada:**
- la matriz $\mathbf{A}$ de tamaño $n$,
- una tolerancia $\tau$

**Salida:**
- El arreglo $\mathbf{p}$ de enteros de tamaño $n$ que tiene la información de una permutación
  realizada a las filas de la matriz, resultado del pivoteo parcial.
- Las matrices  $\mathbf{L}$ y  $\mathbf{U}$ de tamaño $n$.
- Un booleano que es igual a `True` si el algoritmo concluye de manera exitosa,
  o es `False` en caso contrario.

**Nota:**  Hay que tener cuidado al escribir el código porque el algoritmo está
descrito de modo que los índices de las matrices y vectores empiezan en $1$,
mientras que en C empiezan en $0$.

`------------`

2. Escriba la función que resuelve el sistema
   $\mathbf{L}\mathbf{U}\mathbf{x} = \mathbf{P}\mathbf{b}$,
   donde $\mathbf{L}$ es una matriz triangular inferior y
   $\mathbf{U}$ es una matriz triangular superior. 

**Entrada:**
- Las matrices $\mathbf{L}$ y $\mathbf{U}$
- El vector $\mathbf{b}$,
- El arreglo de enteros $\mathbf{p}$,
- Una tolerancia $\tau$.

**Salida:**
- El arreglo $\mathbf{x}$ o `NULL`

La función debe hacer lo siguiente:
- Crear un arreglo $\mathbf{\hat{b}} = (\hat{b}_1, ..., \hat{b}_n)^\top$
  con los elementos de $\mathbf{b} = (b_1, ..., b_n)^\top$ reordenados de
  acuerdo a $\mathbf{p}=(p_1, ..., p_n)^\top$, esto es, $\hat{b}_i = b_{p_i}$.

- Use las funciones `backwardSubstitution` y
  `forwardSubstitution` para resolver el sistema
  de ecuaciones $\mathbf{L}\mathbf{U}\mathbf{x} = \mathbf{\hat{b}}$.
  Si no hay ningún problema, la función debe devolver
  la solución $\mathbf{x}$. En caso contrario, devolver `NULL`.

`------------`


3. Escriba la función recibe el nombre los archivos binarios que corresponden
   a la matriz $\mathbf{A}$ y al vector $\mathbf{b}$ del sistema de ecuaciones
   $\mathbf{A}\mathbf{x}=\mathbf{b}$, lo resuelve usando la factorización LU e
   imprime la información sobre la solución.
   
**Entrada:**
- Nombre del archivo de la matriz $\mathbf{A}$
- Nombre del archivo del vector $\mathbf{b}$
- Una tolerancia $\tau$

La función debe hacer lo siguiente:

- Leer los archivos binarios para crear los arreglos $\mathbf{A}$ y $\mathbf{b}$.
- Imprima el tamaño de la matriz y del vector.
- Usar la función que calcula la factorización LU de $\mathbf{A}$ usando como tolerancia 
  $\tau = \epsilon_m^{2/3}$, donde $\epsilon_m$ es el épsilon de la máquina. 
- Si no logró factorizar la matriz, imprimir un mensaje que indique que la
  matriz es singular y terminar la ejecución.
- En caso contrario, use la función del punto anterior  para resolver
  el sistema de ecuaciones $\mathbf{L}\mathbf{U}\mathbf{x} = \mathbf{P}\mathbf{b}$,
  imprima las primeras y últimas 3 entradas del vector solución y
  el valor del error  $\|\mathbf{A}\mathbf{x} - \mathbf{b}\|$
  

4. Pruebe la función anterior usando los datos en el archivo ``datosTarea03.zip``:
   
| Matriz:      | Vector:  |
|--------------|----------|
| matA005      | vecb005  |
| matA010      | vecb010  |
| matA025      | vecb025  |
| matA050      | vecb050  |
| matA500      | vecb500  |



### Solución:

In [ ]:
# %load ejercicio3/Ejercicio3.c
#include "../arrays/array1D.h"
#include "../arrays/array2D.h"
#include "../sol_ecu_lin/sol_ecu_lin.h"
#include <math.h>

#define OK 1
#define ERROR_METHOD 2
#define ERROR_READ_BIN 4
#define ERROR_INPUT 8

// Devuelve el épsilon de la máquina
double epsilon(void);

int main(int argc, char **argv){

    Array2d *A=NULL;
    Array1d *b=NULL,*x=NULL;
    outLU *resultado=NULL;

    double tol = pow(epsilon(),2.0/3.0);

    if(argc<3) {
        printf("Hay que proporcionar dos para'metros:");
        printf("1. El nombre del archivo binario de un arreglo 1D");
        printf("2. El nombre del archivo binario de un arreglo 2D");
        return ERROR_INPUT;
    } 

    b = readArray1d(argv[1]);
    if(!b){return ERROR_READ_BIN;}
    
    A = readArray2d(argv[2]);
    if(!A){
        freeArray1d(b);
        return ERROR_READ_BIN;
    }

    /*Tamaño de b*/
    printf("\nEl tamano del vector b es <%zu>",b->n);
    /*Elementos de b*/
    // printf("\n");
    // printArray1d(b, "% 6.2f  ", 3);

    /*Tamaño de A*/
    printf("\nLa matriz A tiene <%zu> filas y <%zu> columnas", A->rows, A->cols);
    /*Elementos de U*/
    // printf("\n");
    // printArray2d(U, "% 6.2f  ", 3);

    // Factorización LU con pivoteo parcial
    resultado = LU(A,tol);
    if(!resultado){
        freeArray1d(b); freeArray2d(A);
        return ERROR_METHOD;}

    if(resultado->res==SINGULAR){
        printf("\n%s","La matriz es singular.");
        freeArray2d(A); freeArray1d(b);
        freeArray2d(resultado->L); freeArray2d(resultado->U); freeArray1d(resultado->p); free(resultado);
        return SINGULAR;
    }

    // Sistema LUx = Pb
    x = solveLU(resultado->L,resultado->U,resultado->p,b,tol);
    if(!x){
        freeArray2d(A); freeArray1d(b);
        freeArray2d(resultado->L); freeArray2d(resultado->U); freeArray1d(resultado->p); free(resultado);
        return ERROR_METHOD;
    }

    // Elementos de x
    printf("\nLa solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:\n");
    printArray1d(x, "% 6.2f  ", 3);

    // Error residual
    // Vector residual: Ax-b
    double r_i; /**< Entrada i-ésima del vector residual */
    double err_residual = 0;

    for(size_t i=0;i<A->rows;i++){
        r_i = 0;
        for(size_t j=0;j<A->cols;j++){
            r_i += A->data[i][j] * x->data[j];
        }
        r_i -= b->data[i];
        err_residual += r_i*r_i;
    }
    err_residual = sqrt(err_residual);

    printf("\nError residual ||Ax-b|| = %e",err_residual);

    freeArray2d(A); freeArray1d(b); freeArray1d(x);
    freeArray2d(resultado->L); freeArray2d(resultado->U); freeArray1d(resultado->p); free(resultado);
    return OK;
}

double epsilon(void){
    
    double eps = 0.5;
    double unit = 1.0;
    double val = unit + eps;

    while(val > unit){
        eps/=2;
        val = unit + eps;
    }
    eps = 2*eps;

    return eps;
}

In [64]:
!gcc ejercicio3\Ejercicio3.c arrays\array1D.c arrays\array2D.c sol_ecu_lin\sol_ecu_lin.c -lm -o ejercicio3\ex3

In [65]:
!ejercicio3\ex3 datosTarea03\vecb005.bin datosTarea03\matA005.bin


El tamano del vector b es <5>
La matriz A tiene <5> filas y <5> columnas
La solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:
[  2.20    4.17   -1.36   -2.28    1.34  ]

Error residual ||Ax-b|| = 1.749509e-14


In [66]:
!ejercicio3\ex3 datosTarea03\vecb010.bin datosTarea03\matA010.bin


El tamano del vector b es <10>
La matriz A tiene <10> filas y <10> columnas
La matriz es singular.


In [67]:
!ejercicio3\ex3 datosTarea03\vecb025.bin datosTarea03\matA025.bin


El tamano del vector b es <25>
La matriz A tiene <25> filas y <25> columnas
La matriz es singular.


In [68]:
!ejercicio3\ex3 datosTarea03\vecb050.bin datosTarea03\matA050.bin


El tamano del vector b es <50>
La matriz A tiene <50> filas y <50> columnas
La solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:
[ -5.16   -9.12    9.09   ... -14.54   -5.32    4.74  ]

Error residual ||Ax-b|| = 2.062496e-12


In [69]:
!ejercicio3\ex3 datosTarea03\vecb500.bin datosTarea03\matA500.bin


El tamano del vector b es <500>
La matriz A tiene <500> filas y <500> columnas
La solucio'n del sistema (mostramos los primeros y u'ltimos 3 elementos) es:
[ -1.02   -1.36   -0.01   ...  -0.84   -2.90   -0.46  ]

Error residual ||Ax-b|| = 1.745041e-10
